In [1]:
# ============================================================
# CELDA 1 — Imports y rutas
# ============================================================

import os, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm          # plain tqdm, NO tqdm.notebook

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib
matplotlib.use('Agg')           # backend sin pantalla para Jupyter
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore')

# ── Rutas base ──────────────────────────────────────────────
BASE_DIR = Path('/mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks')
SPATIAL_DIR  = BASE_DIR / 'data/spatial_outputs'
SPATIOTEMP   = SPATIAL_DIR / 'spatiotemporal_outputs'
MANIFEST_02  = SPATIAL_DIR / 'manifest.csv'
OUTPUT_DIR   = SPATIAL_DIR / 'prediction_outputs'
CKPT_04      = OUTPUT_DIR / 'checkpoints_bnn'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_04.mkdir(parents=True, exist_ok=True)

# ── Archivos de entrada (cuaderno 03) ───────────────────────
PATH_EMBEDDINGS = SPATIOTEMP / 'embeddings_spatiotemporal.npy'
PATH_SCORES     = SPATIOTEMP / 'risk_scores.npy'
PATH_MANIFEST03 = SPATIOTEMP / 'scene_manifest.csv'

# ── Semilla y dispositivo ───────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Hiperparámetros globales ────────────────────────────────
BATCH_SIZE  = 8
N_MC        = 50    # pasadas Monte Carlo para BNN
SIGMA_TTA   = 0.01  # std del ruido gaussiano para TTA
N_TTA       = 5     # versiones por secuencia

# ── Verificación ────────────────────────────────────────────
print(f"Dispositivo : {DEVICE}")
print(f"PyTorch     : {torch.__version__}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"CKPT_04     : {CKPT_04}")
print("Archivos de entrada:")
for p in [PATH_EMBEDDINGS, PATH_SCORES, PATH_MANIFEST03, MANIFEST_02]:
    estado = "✔" if p.exists() else "✘ NO ENCONTRADO"
    print(f"  {estado}  {p.name}")


Dispositivo : cuda
PyTorch     : 2.5.1+cu121
OUTPUT_DIR  : /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/prediction_outputs
CKPT_04     : /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/prediction_outputs/checkpoints_bnn
Archivos de entrada:
  ✔  embeddings_spatiotemporal.npy
  ✔  risk_scores.npy
  ✔  scene_manifest.csv
  ✔  manifest.csv


In [2]:
# ============================================================
# CELDA 2 — Cargar embeddings, risk_scores y scene_manifest
# ============================================================

# ── Embeddings del cuaderno 03 ──────────────────────────────
embeddings = np.load(PATH_EMBEDDINGS)   # (12175, 128)
risk_scores = np.load(PATH_SCORES)      # (12175,)

# ── Manifest de secuencias (cuaderno 03) ────────────────────
manifest03 = pd.read_csv(PATH_MANIFEST03)

# ── Verificación de shapes ──────────────────────────────────
print("=== Embeddings ===")
print(f"  shape : {embeddings.shape}")
print(f"  dtype : {embeddings.dtype}")
print(f"  rango : [{embeddings.min():.4f}, {embeddings.max():.4f}]")

print("\n=== Risk scores ===")
print(f"  shape : {risk_scores.shape}")
print(f"  rango : [{risk_scores.min():.4f}, {risk_scores.max():.4f}]")
print(f"  media : {risk_scores.mean():.4f}")
print(f"  std   : {risk_scores.std():.4f}")

print("\n=== scene_manifest (cuaderno 03) ===")
print(f"  filas   : {len(manifest03)}")
print(f"  columnas: {list(manifest03.columns)}")
print(f"  splits  : {manifest03['split'].value_counts().to_dict()}")
print(f"  NaN     : {manifest03.isnull().sum().sum()}")

# ── Sanidad: alineación entre arrays ───────────────────────
assert len(embeddings) == len(risk_scores) == len(manifest03), \
    "❌ Desalineación: embeddings, risk_scores y manifest tienen distinto tamaño"
print("\n✔ Alineación correcta entre embeddings, risk_scores y manifest")


=== Embeddings ===
  shape : (12175, 128)
  dtype : float32
  rango : [0.0000, 34.8229]

=== Risk scores ===
  shape : (12175,)
  rango : [0.0554, 0.8768]
  media : 0.7487
  std   : 0.1540

=== scene_manifest (cuaderno 03) ===
  filas   : 12175
  columnas: ['seq_id', 'frames', 'risk_score', 'split']
  splits  : {'train': 8556, 'val': 1811, 'test': 1808}
  NaN     : 0

✔ Alineación correcta entre embeddings, risk_scores y manifest


In [5]:
# ============================================================
# CELDA 3 — Merge con metadata del cuaderno 02
# ============================================================

# ── Cargar manifest del cuaderno 02 ────────────────────────
manifest02 = pd.read_csv(MANIFEST_02)
print(f"manifest02: {len(manifest02)} imágenes, columnas: {list(manifest02.columns)}")

# ── Extraer el primer frame de cada secuencia como image_id ─
# frames tiene formato "img1|img2|...|img10" — tomamos el primero
manifest03['first_frame'] = manifest03['frames'].str.split('|').str[0].str.strip()

# ── Join: manifest03 ← manifest02[weather, timeofday, num_objects, path_boxes]
meta_cols = ['image_id', 'weather', 'timeofday', 'num_objects', 'path_boxes']
df_meta = manifest02[meta_cols].drop_duplicates('image_id')

df = manifest03.merge(
    df_meta,
    left_on='first_frame',
    right_on='image_id',
    how='left'
)

# ── Rellenar NaN de metadata con valores por defecto ───────
df['weather']     = df['weather'].fillna('undefined')
df['timeofday']   = df['timeofday'].fillna('undefined')
df['num_objects'] = df['num_objects'].fillna(0).astype(int)
df['path_boxes']  = df['path_boxes'].fillna('')

# ── Añadir índice posicional para alinear con arrays numpy ─
df = df.reset_index(drop=True)
df['idx'] = df.index

# ── Verificación ────────────────────────────────────────────
print(f"\ndf final: {len(df)} filas, {len(df.columns)} columnas")
print(f"NaN restantes: {df[['weather','timeofday','num_objects']].isnull().sum().to_dict()}")
print(f"\nweather dist:")
print(df['weather'].value_counts().to_string())
print(f"\ntimeofday dist:")
print(df['timeofday'].value_counts().to_string())

matched = df['image_id'].notna().sum()
print(f"\nSecuencias con metadata encontrada : {matched} / {len(df)}")
print(f"Secuencias sin match (undefined)   : {len(df) - matched}")

manifest02: 61345 imágenes, columnas: ['image_id', 'split', 'weather', 'timeofday', 'clahe_mode', 'num_objects', 'path_curada', 'path_boxes', 'path_mask', 'path_features']

df final: 12175 filas, 11 columnas
NaN restantes: {'weather': 0, 'timeofday': 0, 'num_objects': 0}

weather dist:
weather
clear            6536
overcast         1550
undefined        1408
snowy             957
rainy             862
partly cloudy     850
foggy              12

timeofday dist:
timeofday
daytime      6500
night        4833
dawn/dusk     827
undefined      15

Secuencias con metadata encontrada : 12175 / 12175
Secuencias sin match (undefined)   : 0


In [6]:
# ============================================================
# CELDA 4 — Calcular umbrales por percentil (p80, p95)
# ============================================================

# ── Usar solo el conjunto train para calcular umbrales ──────
train_mask = df['split'] == 'train'
train_scores = risk_scores[df.index[train_mask].values]

p80 = float(np.percentile(train_scores, 80))
p95 = float(np.percentile(train_scores, 95))

print(f"Conjunto train : {train_mask.sum()} secuencias")
print(f"Umbral p80     : {p80:.4f}  → separa low / medium risk")
print(f"Umbral p95     : {p95:.4f}  → separa medium / high risk")

# ── Distribución esperada aplicando umbrales al total ───────
low    = (risk_scores < p80).sum()
medium = ((risk_scores >= p80) & (risk_scores < p95)).sum()
high   = (risk_scores >= p95).sum()

print(f"\nDistribución global (12175 secuencias):")
print(f"  low    (<p80) : {low}  ({100*low/len(risk_scores):.1f}%)")
print(f"  medium (p80-p95): {medium}  ({100*medium/len(risk_scores):.1f}%)")
print(f"  high   (>p95) : {high}  ({100*high/len(risk_scores):.1f}%)")

# ── Guardar para uso en celdas posteriores ──────────────────
THRESHOLDS = {'p80': p80, 'p95': p95}
print(f"\n✔ Umbrales guardados en THRESHOLDS")

Conjunto train : 8556 secuencias
Umbral p80     : 0.8612  → separa low / medium risk
Umbral p95     : 0.8689  → separa medium / high risk

Distribución global (12175 secuencias):
  low    (<p80) : 9742  (80.0%)
  medium (p80-p95): 1816  (14.9%)
  high   (>p95) : 617  (5.1%)

✔ Umbrales guardados en THRESHOLDS


In [7]:
# CELDA 4b — Diagnóstico de distribución de risk_scores
print("Percentiles del conjunto train:")
for p in [10, 25, 50, 75, 80, 90, 95, 99]:
    print(f"  p{p:02d}: {np.percentile(train_scores, p):.4f}")

print(f"\nHistograma aproximado (train):")
counts, edges = np.histogram(train_scores, bins=10)
for i, c in enumerate(counts):
    bar = '█' * (c // 100)
    print(f"  [{edges[i]:.3f}-{edges[i+1]:.3f}]: {bar} {c}")

Percentiles del conjunto train:
  p10: 0.5257
  p25: 0.6913
  p50: 0.8170
  p75: 0.8583
  p80: 0.8612
  p90: 0.8662
  p95: 0.8689
  p99: 0.8725

Histograma aproximado (train):
  [0.055-0.138]:  21
  [0.138-0.220]:  57
  [0.220-0.302]: █ 106
  [0.302-0.384]: █ 165
  [0.384-0.466]: ██ 261
  [0.466-0.548]: ███ 364
  [0.548-0.630]: ██████ 618
  [0.630-0.712]: █████████ 914
  [0.712-0.795]: ███████████ 1110
  [0.795-0.877]: █████████████████████████████████████████████████ 4940


In [9]:
# ============================================================
# CELDA 5 — TTA sobre embeddings (Test Time Augmentation)
# ============================================================

class TTAPredictor:
    """
    TTA sobre vectores de embedding 128-dim.
    Genera N_TTA versiones con ruido gaussiano σ=SIGMA_TTA,
    promedia las predicciones para un risk score más robusto.
    """
    def __init__(self, model, n_tta=N_TTA, sigma=SIGMA_TTA, device=DEVICE):
        self.model  = model
        self.n_tta  = n_tta
        self.sigma  = sigma
        self.device = device

    @torch.no_grad()
    def predict(self, embeddings_batch):
        """
        embeddings_batch: Tensor (B, 128)
        retorna: Tensor (B,) — promedio de N_TTA predicciones
        """
        self.model.eval()
        preds = []
        for _ in range(self.n_tta):
            noise = torch.randn_like(embeddings_batch) * self.sigma
            out   = self.model(embeddings_batch + noise).squeeze(-1)
            preds.append(out)
        return torch.stack(preds, dim=0).mean(dim=0)  # (B,)


# ── Verificación con embeddings sintéticos ──────────────────
# (el modelo BNN aún no existe — usamos una red dummy para probar la clase)
_dummy_model = nn.Sequential(
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 1),   nn.Sigmoid()
).to(DEVICE)

_batch = torch.randn(4, 128).to(DEVICE)
_tta   = TTAPredictor(_dummy_model)
_out   = _tta.predict(_batch)

print(f"TTA input shape  : {_batch.shape}")
print(f"TTA output shape : {_out.shape}")
print(f"TTA output rango : [{_out.min():.4f}, {_out.max():.4f}]")
print(f"n_tta={N_TTA}, sigma={SIGMA_TTA}")
print("✔ TTAPredictor verificado")

del _dummy_model, _batch, _tta, _out

TTA input shape  : torch.Size([4, 128])
TTA output shape : torch.Size([4])
TTA output rango : [0.5742, 0.6046]
n_tta=5, sigma=0.01
✔ TTAPredictor verificado


In [10]:
# ============================================================
# CELDA 6 — BNN con MC Dropout
# ============================================================

class MCDropout(nn.Module):
    """Dropout con training=True forzado en inferencia."""
    def __init__(self, p=0.3):
        super().__init__()
        self.p = p

    def forward(self, x):
        return nn.functional.dropout(x, p=self.p, training=True)


class BNNRiskPredictor(nn.Module):
    """
    Red bayesiana aproximada vía MC Dropout.
    Input : embedding 128-dim (cuaderno 03)
    Output: risk score escalar [0, 1]
    """
    def __init__(self, input_dim=128, hidden_dim=64, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            MCDropout(p=dropout_p),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

    @torch.no_grad()
    def mc_predict(self, x, n_samples=N_MC):
        """
        n_samples pasadas Monte Carlo con dropout activo.
        Retorna media, std e IC 95% por muestra.
        x: Tensor (B, 128)
        retorna: dict con tensores (B,)
        """
        self.train()  # activa dropout
        preds = torch.stack(
            [self(x).squeeze(-1) for _ in range(n_samples)], dim=0
        )  # (n_samples, B)
        self.eval()

        mean = preds.mean(dim=0)
        std  = preds.std(dim=0)
        ci_lower = torch.quantile(preds, 0.025, dim=0)
        ci_upper = torch.quantile(preds, 0.975, dim=0)

        return {
            'mean':     mean,
            'std':      std,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper
        }


# ── Verificación ────────────────────────────────────────────
bnn = BNNRiskPredictor().to(DEVICE)

_batch = torch.randn(4, 128).to(DEVICE)
_out   = bnn.mc_predict(_batch, n_samples=N_MC)

print(f"Parámetros del BNN : {sum(p.numel() for p in bnn.parameters()):,}")
print(f"mc_predict output keys: {list(_out.keys())}")
print(f"mean shape  : {_out['mean'].shape}")
print(f"std  shape  : {_out['std'].shape}")
print(f"mean rango  : [{_out['mean'].min():.4f}, {_out['mean'].max():.4f}]")
print(f"std  rango  : [{_out['std'].min():.4f}, {_out['std'].max():.4f}]")
print("✔ BNNRiskPredictor verificado")

del _batch, _out

Parámetros del BNN : 8,321
mc_predict output keys: ['mean', 'std', 'ci_lower', 'ci_upper']
mean shape  : torch.Size([4])
std  shape  : torch.Size([4])
mean rango  : [0.4061, 0.4562]
std  rango  : [0.0327, 0.0447]
✔ BNNRiskPredictor verificado


In [11]:
# ============================================================
# CELDA 7 — Entrenamiento BNN
# ============================================================

# ── Preparar tensores por split ─────────────────────────────
def make_tensors(split_name):
    mask = df['split'] == split_name
    idx  = df.index[mask].values
    X    = torch.tensor(embeddings[idx], dtype=torch.float32)
    y    = torch.tensor(risk_scores[idx], dtype=torch.float32)
    return TensorDataset(X, y)

ds_train = make_tensors('train')
ds_val   = make_tensors('val')

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(ds_train)} secuencias, {len(dl_train)} batches")
print(f"Val  : {len(ds_val)}  secuencias, {len(dl_val)} batches")

# ── Optimizador y loss ──────────────────────────────────────
optimizer = torch.optim.AdamW(bnn.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

# ── Early stopping ──────────────────────────────────────────
PATIENCE   = 5
best_val   = float('inf')
patience_c = 0
best_ckpt  = CKPT_04 / 'bnn_best.pt'

# ── Loop de entrenamiento ───────────────────────────────────
print("\nIniciando entrenamiento BNN...")
for epoch in range(1, 51):
    # — train —
    bnn.train()
    train_loss = 0.0
    for X_b, y_b in dl_train:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        pred = bnn(X_b).squeeze(-1)
        loss = criterion(pred, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bnn.parameters(), max_norm=0.5)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(dl_train)

    # — val —
    bnn.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, y_b in dl_val:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            pred = bnn(X_b).squeeze(-1)
            val_loss += criterion(pred, y_b).item()
    val_loss /= len(dl_val)

    if epoch % 5 == 0 or epoch == 1:
        print(f"  Época {epoch:02d} — train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}")

    # — early stopping —
    if val_loss < best_val:
        best_val   = val_loss
        patience_c = 0
        torch.save(bnn.state_dict(), best_ckpt)
    else:
        patience_c += 1
        if patience_c >= PATIENCE:
            print(f"\n  Early stopping en época {epoch} (mejor val_loss: {best_val:.4f})")
            break

# ── Cargar mejor modelo ─────────────────────────────────────
bnn.load_state_dict(torch.load(best_ckpt, weights_only=True))
bnn.eval()
print(f"\n✔ Mejor modelo cargado desde {best_ckpt.name}")
print(f"  best val_loss: {best_val:.4f}")

Train: 8556 secuencias, 1070 batches
Val  : 1811  secuencias, 227 batches

Iniciando entrenamiento BNN...
  Época 01 — train_loss: 0.0203  val_loss: 0.0170
  Época 05 — train_loss: 0.0158  val_loss: 0.0155

  Early stopping en época 9 (mejor val_loss: 0.0148)

✔ Mejor modelo cargado desde bnn_best.pt
  best val_loss: 0.0148


In [12]:
# ============================================================
# CELDA 8 — Inferencia con incertidumbre (MC Dropout + TTA)
# ============================================================

tta_predictor = TTAPredictor(bnn, n_tta=N_TTA, sigma=SIGMA_TTA, device=DEVICE)

all_mean     = []
all_std      = []
all_ci_lower = []
all_ci_upper = []
all_tta      = []

# ── Inferencia sobre todas las 12175 secuencias ─────────────
X_all = torch.tensor(embeddings, dtype=torch.float32)
ds_all = TensorDataset(X_all)
dl_all = DataLoader(ds_all, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

bnn.eval()
with torch.no_grad():
    for (X_b,) in tqdm(dl_all, desc='Inferencia BNN+TTA'):
        X_b = X_b.to(DEVICE)

        # MC Dropout — incertidumbre
        mc_out = bnn.mc_predict(X_b, n_samples=N_MC)
        all_mean.append(mc_out['mean'].cpu())
        all_std.append(mc_out['std'].cpu())
        all_ci_lower.append(mc_out['ci_lower'].cpu())
        all_ci_upper.append(mc_out['ci_upper'].cpu())

        # TTA — robustez
        tta_out = tta_predictor.predict(X_b)
        all_tta.append(tta_out.cpu())

# ── Concatenar resultados ───────────────────────────────────
pred_mean     = torch.cat(all_mean).numpy()
pred_std      = torch.cat(all_std).numpy()
pred_ci_lower = torch.cat(all_ci_lower).numpy()
pred_ci_upper = torch.cat(all_ci_upper).numpy()
pred_tta      = torch.cat(all_tta).numpy()

# ── Score final: promedio MC mean + TTA ────────────────────
pred_final = (pred_mean + pred_tta) / 2.0

# ── Verificación ────────────────────────────────────────────
print(f"pred_final shape : {pred_final.shape}")
print(f"pred_final rango : [{pred_final.min():.4f}, {pred_final.max():.4f}]")
print(f"pred_final media : {pred_final.mean():.4f}")
print(f"incertidumbre std media : {pred_std.mean():.4f}")
print(f"IC 95% ancho medio      : {(pred_ci_upper - pred_ci_lower).mean():.4f}")
print("✔ Inferencia completada")

Inferencia BNN+TTA: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 1522/1522 [00:13<00:00, 113.32it/s]

pred_final shape : (12175,)
pred_final rango : [0.5177, 0.8896]
pred_final media : 0.7476
incertidumbre std media : 0.0251
IC 95% ancho medio      : 0.0880
✔ Inferencia completada


In [13]:
# ============================================================
# CELDA 9 — Asignar risk_level por umbral percentil
# ============================================================

def asignar_nivel(scores, p80, p95):
    niveles = np.where(
        scores >= p95, 'high',
        np.where(scores >= p80, 'medium', 'low')
    )
    return niveles

df['pred_final']  = pred_final
df['pred_std']    = pred_std
df['ci_lower']    = pred_ci_lower
df['ci_upper']    = pred_ci_upper
df['risk_level']  = asignar_nivel(pred_final, p80, p95)

# ── Verificación ────────────────────────────────────────────
dist = df['risk_level'].value_counts()
print(f"Distribución de risk_level (pred_final):")
for nivel in ['low', 'medium', 'high']:
    n = dist.get(nivel, 0)
    print(f"  {nivel:6s}: {n:5d}  ({100*n/len(df):.1f}%)")

print(f"\nUmbrales aplicados: p80={p80:.4f}, p95={p95:.4f}")
print(f"Nota: umbrales calculados sobre train ({train_mask.sum()} secuencias)")

# ── Verificar por split ─────────────────────────────────────
print(f"\nDistribución por split:")
print(df.groupby('split')['risk_level'].value_counts().unstack(fill_value=0).to_string())

print("\n✔ risk_level asignado a todas las secuencias")

Distribución de risk_level (pred_final):
  low   : 11969  (98.3%)
  medium:   135  (1.1%)
  high  :    71  (0.6%)

Umbrales aplicados: p80=0.8612, p95=0.8689
Nota: umbrales calculados sobre train (8556 secuencias)

Distribución por split:
risk_level  high   low  medium
split                         
test           7  1783      18
train         52  8407      97
val           12  1779      20

✔ risk_level asignado a todas las secuencias


In [14]:
# ============================================================
# CELDA 9 — Asignar risk_level (sobre risk_scores originales)
# ============================================================

# Los umbrales p80/p95 se calcularon sobre risk_scores del cuaderno 03.
# pred_final del BNN se usa para incertidumbre y CI, no para clasificación,
# porque la regresión a la media comprime las predicciones bajo los umbrales.

df['pred_final']      = pred_final
df['pred_std']        = pred_std
df['ci_lower']        = pred_ci_lower
df['ci_upper']        = pred_ci_upper
df['risk_score_orig'] = risk_scores   # score original del cuaderno 03

df['risk_level'] = asignar_nivel(risk_scores, p80, p95)

# ── Verificación ────────────────────────────────────────────
dist = df['risk_level'].value_counts()
print(f"Distribución de risk_level (risk_scores originales):")
for nivel in ['low', 'medium', 'high']:
    n = dist.get(nivel, 0)
    print(f"  {nivel:6s}: {n:5d}  ({100*n/len(df):.1f}%)")

print(f"\nUmbrales: p80={p80:.4f}, p95={p95:.4f}")

print(f"\nDistribución por split:")
print(df.groupby('split')['risk_level'].value_counts().unstack(fill_value=0).to_string())

print(f"\nCorrelación pred_final vs risk_score_orig: "
      f"{np.corrcoef(pred_final, risk_scores)[0,1]:.4f}")
print("✔ risk_level asignado desde risk_scores originales")

Distribución de risk_level (risk_scores originales):
  low   :  9742  (80.0%)
  medium:  1816  (14.9%)
  high  :   617  (5.1%)

Umbrales: p80=0.8612, p95=0.8689

Distribución por split:
risk_level  high   low  medium
split                         
test          93  1440     275
train        428  6844    1284
val           96  1458     257

Correlación pred_final vs risk_score_orig: 0.6102
✔ risk_level asignado desde risk_scores originales


In [15]:
# ============================================================
# CELDA 10 — Risk Zone estimation desde bounding boxes
# ============================================================

def cargar_boxes(path_boxes):
    """Carga detecciones YOLO desde JSON. Retorna lista de boxes o []."""
    if not path_boxes or not os.path.exists(path_boxes):
        return []
    try:
        with open(path_boxes, 'r') as f:
            data = json.load(f)
        return data.get('boxes', [])
    except Exception:
        return []

def expandir_zona(boxes, risk_score, img_w=416, img_h=416):
    """
    Expande cada bounding box según risk_score.
    expansion_factor = risk_score → mayor riesgo = zona más grande.
    Retorna lista de zonas expandidas con coordenadas normalizadas [0,1].
    """
    zonas = []
    factor = float(risk_score)
    for box in boxes:
        x1, y1, x2, y2 = box.get('bbox_xyxy', [0, 0, 0, 0])
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        w  = (x2 - x1) * (1 + factor)
        h  = (y2 - y1) * (1 + factor)
        # expandir desde el centro, clamp a dimensiones de imagen
        nx1 = max(0, cx - w / 2)
        ny1 = max(0, cy - h / 2)
        nx2 = min(img_w, cx + w / 2)
        ny2 = min(img_h, cy + h / 2)
        zonas.append({
            'class_name':  box.get('class_name', 'unknown'),
            'confidence':  round(float(box.get('confidence', 0)), 4),
            'bbox_orig':   [round(x1,1), round(y1,1), round(x2,1), round(y2,1)],
            'bbox_expanded': [round(nx1,1), round(ny1,1), round(nx2,1), round(ny2,1)],
            'expansion_factor': round(factor, 4)
        })
    return zonas

# ── Aplicar a todas las secuencias ─────────────────────────
risk_zones_all = []
n_sin_boxes = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc='Risk zones'):
    boxes = cargar_boxes(row['path_boxes'])
    if not boxes:
        n_sin_boxes += 1
    zonas = expandir_zona(boxes, row['risk_score_orig'])
    risk_zones_all.append(zonas)

df['risk_zones'] = risk_zones_all

# ── Verificación ────────────────────────────────────────────
n_con_boxes = len(df) - n_sin_boxes
print(f"Secuencias con bounding boxes : {n_con_boxes} ({100*n_con_boxes/len(df):.1f}%)")
print(f"Secuencias sin boxes          : {n_sin_boxes} ({100*n_sin_boxes/len(df):.1f}%)")

# Ejemplo de zona expandida
ejemplo = df[df['risk_zones'].map(len) > 0].iloc[0]
print(f"\nEjemplo — seq_id: {ejemplo['seq_id']}")
print(f"  risk_score : {ejemplo['risk_score_orig']:.4f}")
print(f"  n_zonas    : {len(ejemplo['risk_zones'])}")
print(f"  primera zona: {ejemplo['risk_zones'][0]}")
print("\n✔ Risk zones calculadas")

Risk zones: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 12175/12175 [00:04<00:00, 2591.63it/s]


Secuencias con bounding boxes : 11232 (92.3%)
Secuencias sin boxes          : 943 (7.7%)

Ejemplo — seq_id: train_clear_dawn/dusk
  risk_score : 0.8595
  n_zonas    : 1
  primera zona: {'class_name': 'person', 'confidence': 0.2801, 'bbox_orig': [251.0, 213.8, 259.7, 241.1], 'bbox_expanded': [247.3, 202.0, 263.4, 252.9], 'expansion_factor': 0.8595}

✔ Risk zones calculadas


In [16]:
# ============================================================
# CELDA 11 — Generar prediction.json
# ============================================================

predictions = []
timestamp_now = datetime.utcnow().isoformat() + 'Z'

for i, row in tqdm(df.iterrows(), total=len(df), desc='Generando prediction.json'):
    predictions.append({
        'scene_id':   str(row['seq_id']),
        'seq_id':     str(row['seq_id']),
        'frames':     str(row['frames']).split('|'),
        'risk_score': round(float(row['risk_score_orig']), 6),
        'uncertainty': round(float(row['pred_std']), 6),
        'confidence_interval': {
            'lower_2.5': round(float(row['ci_lower']), 6),
            'upper_97.5': round(float(row['ci_upper']), 6)
        },
        'risk_level':  str(row['risk_level']),
        'weather':     str(row['weather']),
        'timeofday':   str(row['timeofday']),
        'risk_zones':  row['risk_zones']
    })

out_path = OUTPUT_DIR / 'prediction.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

# ── Verificación ────────────────────────────────────────────
size_mb = out_path.stat().st_size / 1024**2
print(f"prediction.json guardado: {out_path}")
print(f"  registros : {len(predictions)}")
print(f"  tamaño    : {size_mb:.2f} MB")

# Muestra un registro high risk como ejemplo
high_ex = next((p for p in predictions if p['risk_level'] == 'high'), None)
if high_ex:
    print(f"\nEjemplo high risk:")
    print(f"  scene_id   : {high_ex['scene_id']}")
    print(f"  risk_score : {high_ex['risk_score']}")
    print(f"  uncertainty: {high_ex['uncertainty']}")
    print(f"  CI 95%     : [{high_ex['confidence_interval']['lower_2.5']}, {high_ex['confidence_interval']['upper_97.5']}]")
    print(f"  weather    : {high_ex['weather']}")
    print(f"  n_zonas    : {len(high_ex['risk_zones'])}")

print("\n✔ prediction.json generado")

Generando prediction.json: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 12175/12175 [00:00<00:00, 14017.81it/s]


prediction.json guardado: /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/prediction_outputs/prediction.json
  registros : 12175
  tamaño    : 27.80 MB

Ejemplo high risk:
  scene_id   : train_clear_dawn/dusk
  risk_score : 0.869498
  uncertainty: 0.032451
  CI 95%     : [0.788253, 0.903693]
  weather    : clear
  n_zonas    : 4

✔ prediction.json generado


In [17]:
# ============================================================
# CELDA 12 — Generar alerts.csv (medium + high risk)
# ============================================================

df_alerts = df[df['risk_level'].isin(['medium', 'high'])].copy()

# ── Calcular num_objects_avg por secuencia ──────────────────
# num_objects es del primer frame — usamos como aproximación
df_alerts['num_objects_avg'] = df_alerts['num_objects'].astype(float)

# ── Construir DataFrame de alertas ─────────────────────────
alerts = df_alerts[[
    'seq_id', 'risk_score_orig', 'pred_std',
    'risk_level', 'weather', 'timeofday',
    'num_objects_avg', 'frames', 'split'
]].rename(columns={
    'risk_score_orig': 'risk_score',
    'pred_std':        'uncertainty'
})

alerts['scene_id']  = alerts['seq_id']
alerts['timestamp'] = datetime.utcnow().isoformat() + 'Z'

# Ordenar por risk_score descendente
alerts = alerts.sort_values('risk_score', ascending=False).reset_index(drop=True)

# Columnas finales en orden especificado
alerts = alerts[[
    'scene_id', 'seq_id', 'risk_score', 'uncertainty',
    'risk_level', 'weather', 'timeofday',
    'num_objects_avg', 'frames', 'timestamp', 'split'
]]

out_path = OUTPUT_DIR / 'alerts.csv'
alerts.to_csv(out_path, index=False, encoding='utf-8')

# ── Verificación ────────────────────────────────────────────
print(f"alerts.csv guardado: {out_path}")
print(f"  total alertas : {len(alerts)}")
print(f"  high risk     : {(alerts['risk_level'] == 'high').sum()}")
print(f"  medium risk   : {(alerts['risk_level'] == 'medium').sum()}")
print(f"\nAlertas por weather:")
print(alerts['weather'].value_counts().to_string())
print(f"\nAlertas por timeofday:")
print(alerts['timeofday'].value_counts().to_string())
print(f"\nTop 5 escenas de mayor riesgo:")
print(alerts[['scene_id','risk_score','risk_level','weather','timeofday']].head().to_string(index=False))
print("\n✔ alerts.csv generado")

alerts.csv guardado: /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/prediction_outputs/alerts.csv
  total alertas : 2433
  high risk     : 617
  medium risk   : 1816

Alertas por weather:
weather
clear            1160
overcast          458
undefined         263
partly cloudy     217
rainy             169
snowy             164
foggy               2

Alertas por timeofday:
timeofday
daytime      1584
night         674
dawn/dusk     172
undefined       3

Top 5 escenas de mayor riesgo:
               scene_id  risk_score risk_level   weather timeofday
     test_snowy_daytime    0.876781       high     snowy   daytime
  test_overcast_daytime    0.876754       high  overcast   daytime
    train_clear_daytime    0.876722       high     clear   daytime
train_undefined_daytime    0.876431       high undefined   daytime
train_undefined_daytime    0.876017       high undefined   daytime

✔ alerts.csv generado


In [18]:
# ============================================================
# CELDA 13 — Generar risk_map.html
# ============================================================

import base64
from io import BytesIO

def fig_to_base64(fig):
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight',
                facecolor='white')
    buf.seek(0)
    return base64.b64encode(buf.read()).decode('utf-8')

# ── Figura 1: Heatmap risk_score medio por weather × timeofday ─
weather_order   = ['clear','overcast','partly cloudy','rainy','snowy','foggy','undefined']
timeofday_order = ['daytime','night','dawn/dusk','undefined']

pivot = df.groupby(['weather','timeofday'])['risk_score_orig'].mean().unstack(fill_value=0)
pivot = pivot.reindex(index=weather_order, columns=timeofday_order, fill_value=0)

fig1, ax1 = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd',
            vmin=0.5, vmax=0.9, ax=ax1, linewidths=0.5)
ax1.set_title('Risk score medio por condición climática y momento del día', fontsize=11)
ax1.set_xlabel('Momento del día')
ax1.set_ylabel('Clima')
plt.tight_layout()
img1 = fig_to_base64(fig1)
plt.close(fig1)

# ── Figura 2: Distribución de risk_levels por weather ──────
risk_counts = df.groupby(['weather','risk_level']).size().unstack(fill_value=0)
risk_counts = risk_counts.reindex(index=weather_order,
             columns=['low','medium','high'], fill_value=0)

fig2, ax2 = plt.subplots(figsize=(7, 4))
risk_counts.plot(kind='bar', ax=ax2, color=['#4CAF50','#FF9800','#F44336'],
                 edgecolor='white', linewidth=0.5)
ax2.set_title('Distribución de niveles de riesgo por condición climática', fontsize=11)
ax2.set_xlabel('Clima')
ax2.set_ylabel('Número de secuencias')
ax2.legend(title='Nivel')
ax2.tick_params(axis='x', rotation=30)
plt.tight_layout()
img2 = fig_to_base64(fig2)
plt.close(fig2)

# ── Figura 3: Top 10 escenas de mayor riesgo ───────────────
top10 = df.nlargest(10, 'risk_score_orig')[
    ['seq_id','risk_score_orig','pred_std','risk_level','weather','timeofday']
].reset_index(drop=True)

fig3, ax3 = plt.subplots(figsize=(7, 4))
colors = ['#F44336' if r == 'high' else '#FF9800' for r in top10['risk_level']]
bars = ax3.barh(range(len(top10)), top10['risk_score_orig'], color=colors)
ax3.set_yticks(range(len(top10)))
ax3.set_yticklabels([f"{r['seq_id'][:30]}..." if len(r['seq_id']) > 30
                     else r['seq_id'] for _, r in top10.iterrows()], fontsize=8)
ax3.set_xlabel('Risk score')
ax3.set_title('Top 10 escenas de mayor riesgo', fontsize=11)
ax3.set_xlim(0.85, 0.90)
for bar, (_, row) in zip(bars, top10.iterrows()):
    ax3.text(bar.get_width() + 0.0002, bar.get_y() + bar.get_height()/2,
             f"σ={row['pred_std']:.3f}", va='center', fontsize=7)
plt.tight_layout()
img3 = fig_to_base64(fig3)
plt.close(fig3)

# ── Ensamblar HTML ──────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>Risk Map — Equipo8-Grupo8-SIC-2025</title>
<style>
  body {{ font-family: Arial, sans-serif; margin: 0; padding: 24px;
          background: #f5f5f5; color: #222; }}
  h1   {{ font-size: 22px; margin-bottom: 4px; }}
  .sub {{ font-size: 13px; color: #666; margin-bottom: 24px; }}
  .stats {{ display: flex; gap: 16px; margin-bottom: 24px; flex-wrap: wrap; }}
  .card {{ background: white; border-radius: 8px; padding: 16px 20px;
           min-width: 140px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
  .card .val {{ font-size: 28px; font-weight: bold; }}
  .card .lbl {{ font-size: 12px; color: #888; margin-top: 4px; }}
  .low    {{ color: #4CAF50; }} .medium {{ color: #FF9800; }}
  .high   {{ color: #F44336; }}
  .plot   {{ background: white; border-radius: 8px; padding: 16px;
             margin-bottom: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }}
  .plot img {{ width: 100%; max-width: 700px; display: block; margin: auto; }}
  .footer {{ font-size: 11px; color: #aaa; margin-top: 32px; }}
</style>
</head>
<body>
<h1>Mapa de Riesgo de Accidente</h1>
<div class="sub">Equipo8-Grupo8-SIC-2025 · Pipeline SIC 2025 ·
Generado: {datetime.utcnow().strftime('%Y-%m-%d %H:%M')} UTC</div>

<div class="stats">
  <div class="card"><div class="val">{len(df):,}</div>
    <div class="lbl">Secuencias analizadas</div></div>
  <div class="card"><div class="val low">{(df['risk_level']=='low').sum():,}</div>
    <div class="lbl">Low risk</div></div>
  <div class="card"><div class="val medium">{(df['risk_level']=='medium').sum():,}</div>
    <div class="lbl">Medium risk</div></div>
  <div class="card"><div class="val high">{(df['risk_level']=='high').sum():,}</div>
    <div class="lbl">High risk</div></div>
  <div class="card"><div class="val">{THRESHOLDS['p80']:.4f}</div>
    <div class="lbl">Umbral p80</div></div>
  <div class="card"><div class="val">{THRESHOLDS['p95']:.4f}</div>
    <div class="lbl">Umbral p95</div></div>
</div>

<div class="plot"><img src="data:image/png;base64,{img1}" alt="heatmap"></div>
<div class="plot"><img src="data:image/png;base64,{img2}" alt="barras"></div>
<div class="plot"><img src="data:image/png;base64,{img3}" alt="top10"></div>

<div class="footer">Umbrales calculados sobre conjunto train
({train_mask.sum()} secuencias). BNN val_loss={best_val:.4f}.
Correlación BNN vs GCN+LSTM: {np.corrcoef(pred_final, risk_scores)[0,1]:.4f}</div>
</body></html>"""

out_path = OUTPUT_DIR / 'risk_map.html'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"risk_map.html guardado: {out_path}")
print(f"  tamaño: {out_path.stat().st_size/1024:.1f} KB")
print("✔ risk_map.html generado")

risk_map.html guardado: /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/prediction_outputs/risk_map.html
  tamaño: 203.1 KB
✔ risk_map.html generado


In [20]:
# ============================================================
# CELDA 14 — Generar metrics_report.json
# ============================================================
from scipy.stats import pearsonr

# ── Métricas sobre test set ─────────────────────────────────
test_mask  = df['split'] == 'test'
test_idx   = df.index[test_mask].values
y_true     = risk_scores[test_idx]
y_pred_bnn = pred_final[test_idx]

mae  = float(np.abs(y_true - y_pred_bnn).mean())
mse  = float(((y_true - y_pred_bnn) ** 2).mean())
r, _ = pearsonr(y_true, y_pred_bnn)

print(f"=== Métricas test set ({test_mask.sum()} secuencias) ===")
print(f"  MAE        : {mae:.4f}")
print(f"  MSE        : {mse:.4f}")
print(f"  Pearson r  : {r:.4f}")

# ── Distribución de risk_levels ─────────────────────────────
level_counts = df['risk_level'].value_counts().to_dict()
level_pct    = {k: round(100*v/len(df), 2) for k, v in level_counts.items()}

# ── Alertas por weather y timeofday ────────────────────────
alerts_weather   = df_alerts['weather'].value_counts().to_dict()
alerts_timeofday = df_alerts['timeofday'].value_counts().to_dict()

# ── Incertidumbre global ────────────────────────────────────
uncertainty_stats = {
    'mean': round(float(pred_std.mean()), 6),
    'std':  round(float(pred_std.std()),  6),
    'min':  round(float(pred_std.min()),  6),
    'max':  round(float(pred_std.max()),  6),
    'ic95_ancho_medio': round(float((pred_ci_upper - pred_ci_lower).mean()), 6)
}

# ── Construir reporte ───────────────────────────────────────
report = {
    'proyecto':   'Equipo8-Grupo8-SIC-2025',
    'cuaderno':   '04_prediction_alert',
    'generado':   datetime.utcnow().isoformat() + 'Z',
    'dataset': {
        'total_secuencias': len(df),
        'splits': df['split'].value_counts().to_dict(),
        'secuencias_con_boxes': int((df['risk_zones'].map(len) > 0).sum()),
        'secuencias_sin_boxes': int((df['risk_zones'].map(len) == 0).sum())
    },
    'umbrales': {
        'nota': 'Calculados sobre conjunto train por percentil',
        'p80':  round(p80, 6),
        'p95':  round(p95, 6)
    },
    'metricas_test': {
        'n_secuencias': int(test_mask.sum()),
        'MAE':       round(mae, 6),
        'MSE':       round(mse, 6),
        'Pearson_r': round(float(r), 6),
        'nota': 'BNN pred_final vs risk_scores GCN+LSTM'
    },
    'bnn': {
        'arquitectura':  'FC(128→64) + MCDropout(0.3) + FC(64→1) + Sigmoid',
        'n_mc_samples':  N_MC,
        'n_tta':         N_TTA,
        'sigma_tta':     SIGMA_TTA,
        'best_val_loss': round(best_val, 6),
        'correlacion_bnn_vs_gcnlstm': round(float(np.corrcoef(pred_final, risk_scores)[0,1]), 4)
    },
    'distribucion_risk_levels': {
        'conteo':      level_counts,
        'porcentaje':  level_pct
    },
    'alertas': {
        'total':      len(df_alerts),
        'high':       int((df_alerts['risk_level'] == 'high').sum()),
        'medium':     int((df_alerts['risk_level'] == 'medium').sum()),
        'por_weather':   alerts_weather,
        'por_timeofday': alerts_timeofday
    },
    'incertidumbre': uncertainty_stats,
    'outputs': {
        'prediction_json': str(OUTPUT_DIR / 'prediction.json'),
        'alerts_csv':      str(OUTPUT_DIR / 'alerts.csv'),
        'risk_map_html':   str(OUTPUT_DIR / 'risk_map.html'),
        'metrics_report':  str(OUTPUT_DIR / 'metrics_report.json')
    }
}

out_path = OUTPUT_DIR / 'metrics_report.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"\nmetrics_report.json guardado: {out_path}")
print(f"  tamaño: {out_path.stat().st_size/1024:.1f} KB")
print("\n=== Resumen final del pipeline ===")
print(f"  Secuencias procesadas : {len(df):,}")
print(f"  Alertas generadas     : {len(df_alerts):,}  (medium+high)")
print(f"  MAE test              : {mae:.4f}")
print(f"  Pearson r test        : {r:.4f}")
print("\n✔ metrics_report.json generado")
print("✔ Cuaderno 04 completado")

=== Métricas test set (1808 secuencias) ===
  MAE        : 0.0881
  MSE        : 0.0162
  Pearson r  : 0.6153

metrics_report.json guardado: /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/prediction_outputs/metrics_report.json
  tamaño: 2.1 KB

=== Resumen final del pipeline ===
  Secuencias procesadas : 12,175
  Alertas generadas     : 2,433  (medium+high)
  MAE test              : 0.0881
  Pearson r test        : 0.6153

✔ metrics_report.json generado
✔ Cuaderno 04 completado
